# Hot Hand Fallacy — Baseline Analysis & Cognitive Engines
**CS6795 Term Project | Jason Schwartz | Georgia Tech OMSCS**

This notebook completes Tasks 7–12 of the research plan:
- Task 7–8: Data cleaning and baseline statistical analysis
- Task 11: Rational Engine (Bayesian) — Type 2 System
- Task 12: Cognitive Engine (Exponential Decay) — Type 1 System

**Output:** `luka_2025_26_shots_with_beliefs.csv` — the original shot data enriched with both engines' probability estimates at each shot.

## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

# Import the two cognitive engines
from rational_engine import RationalEngine
from cognitive_engine import CognitiveEngine

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12

SHOT_DATA_PATH = 'luka_2025_26_shots.csv'
OUTPUT_PATH    = 'luka_2025_26_shots_with_beliefs.csv'

## 1. Load & Clean Data (Tasks 7–8)

In [3]:
df = pd.read_csv(SHOT_DATA_PATH)
print(f'Raw shape: {df.shape}')
df.head()

Raw shape: (1168, 21)


,game_id,game_date,shot_number,game_shot_number,period,minutes_remaining,seconds_remaining,action_type,shot_type,shot_distance_ft,...,shot_zone_area,shot_zone_range,loc_x,loc_y,shot_made,event_type,streak,rolling_3_fg_pct,rolling_5_fg_pct,rolling_10_fg_pct
0,22500002,2025-10-21,1,1,1,11,39,Fadeaway Jump Shot,2PT Field Goal,13,...,Center(C),8-16 ft.,-60,125,1,Made Shot,0,NaN,NaN,NaN
1,22500002,2025-10-21,2,2,1,10,18,Step Back Jump shot,3PT Field Goal,25,...,Center(C),24+ ft.,18,259,0,Missed Shot,1,1.000000,1.000000,1.000000
2,22500002,2025-10-21,3,3,1,9,7,Jump Shot,3PT Field Goal,26,...,Left Side Center(LC),24+ ft.,-88,248,1,Made Shot,-1,0.500000,0.500000,0.500000
3,22500002,2025-10-21,4,4,1,7,36,Step Back Jump shot,3PT Field Goal,27,...,Left Side Center(LC),24+ ft.,-167,217,0,Missed Shot,1,0.666667,0.666667,0.666667
4,22500002,2025-10-21,5,5,1,5,23,Running Layup Shot,2PT Field Goal,2,...,Center(C),Less Than 8 ft.,2,28,0,Missed Shot,-1,0.333333,0.500000,0.500000


In [4]:
# ── Data Quality Checks ───────────────────────────────────────────────────

print('=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

print('\n=== shot_made distribution (should only be 0 and 1) ===')
print(df['shot_made'].value_counts())

print('\n=== shot_type distribution ===')
print(df['shot_type'].value_counts())

print('\n=== Shots per game (first 5 games) ===')
print(df.groupby('game_date')['shot_made'].count().head())

=== Missing Values ===
rolling_3_fg_pct     1
rolling_5_fg_pct     1
rolling_10_fg_pct    1
dtype: int64

=== shot_made distribution (should only be 0 and 1) ===
shot_made
0    615
1    553
Name: count, dtype: int64

=== shot_type distribution ===
shot_type
2PT Field Goal    615
3PT Field Goal    553
Name: count, dtype: int64

=== Shots per game (first 5 games) ===
game_date
2025-10-21    27
2025-10-24    23
2025-10-31    27
2025-11-02    22
2025-11-05    27
Name: shot_made, dtype: int64


In [5]:
# ── Cleaning Steps ────────────────────────────────────────────────────────

# 1. Drop any rows with null shot outcomes (shouldn't exist, but verify)
before = len(df)
df = df.dropna(subset=['shot_made', 'loc_x', 'loc_y', 'game_id'])
print(f'Dropped {before - len(df)} rows with null required fields')

# 2. Ensure correct types
df['shot_made']  = df['shot_made'].astype(int)
df['game_id']    = df['game_id'].astype(str)
df['game_date']  = pd.to_datetime(df['game_date'])

# 3. Sort into true chronological order (critical for sequence analysis)
df = df.sort_values(['game_date', 'game_id', 'game_shot_number']).reset_index(drop=True)
df['shot_number'] = df.index + 1  # re-index after sort

print(f'\nFinal clean shape: {df.shape}')
print(f'Date range: {df["game_date"].min().date()} → {df["game_date"].max().date()}')
print(f'Games: {df["game_id"].nunique()}')
print(f'Total FGAs: {len(df)}')

Dropped 0 rows with null required fields

Final clean shape: (1168, 21)
Date range: 2025-10-21 → 2026-03-10
Games: 53
Total FGAs: 1168


## 2. Baseline Statistical Analysis (Task 8)

This section establishes the **rational prior** — Luka's true long-term shooting probability that the Bayesian engine will be anchored to.

In [6]:
# ── Overall FG% ───────────────────────────────────────────────────────────

total_fga  = len(df)
total_fgm  = df['shot_made'].sum()
overall_fg = total_fgm / total_fga

# Wilson score confidence interval (better than normal approx for proportions)
ci_low, ci_high = stats.proportion_confint(total_fgm, total_fga, alpha=0.05, method='wilson')

print('=' * 50)
print(f'  Luka Doncic — 2025-26 Regular Season')
print('=' * 50)
print(f'  Total FGA : {total_fga}')
print(f'  Total FGM : {total_fgm}')
print(f'  FG%       : {overall_fg:.4f}  ({overall_fg*100:.1f}%)')
print(f'  95% CI    : [{ci_low:.4f}, {ci_high:.4f}]')
print(f'\n  → Rational Prior for Bayesian Engine: α={total_fgm}, β={total_fga - total_fgm}')

AttributeError: module 'scipy.stats' has no attribute 'proportion_confint'

In [ ]:
# ── FG% by Shot Type ─────────────────────────────────────────────────────

by_type = df.groupby('shot_type')['shot_made'].agg(['sum','count','mean'])
by_type.columns = ['FGM', 'FGA', 'FG%']
by_type['FG%'] = by_type['FG%'].map('{:.1%}'.format)
print('\nFG% by Shot Type:')
print(by_type.to_string())

In [ ]:
# ── FG% by Zone ──────────────────────────────────────────────────────────

by_zone = df.groupby('shot_zone_basic')['shot_made'].agg(['sum','count','mean'])
by_zone.columns = ['FGM', 'FGA', 'FG%']
by_zone = by_zone.sort_values('FGA', ascending=False)
by_zone['FG%'] = by_zone['FG%'].map('{:.1%}'.format)
print('FG% by Zone:')
print(by_zone.to_string())

In [ ]:
# ── Streak Distribution ───────────────────────────────────────────────────
# Key diagnostic: how often does Luka enter a shot on a make/miss streak?

streak_counts = df['streak'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of streak distribution
colors = ['#d7191c' if x < 0 else '#2b83ba' if x > 0 else '#aaaaaa'
          for x in streak_counts.index]
axes[0].bar(streak_counts.index, streak_counts.values, color=colors, edgecolor='white')
axes[0].set_xlabel('Streak entering shot (+ = make streak, - = miss streak)')
axes[0].set_ylabel('Number of shots')
axes[0].set_title('Streak Distribution — All FGAs')
axes[0].axvline(0, color='black', linewidth=1, linestyle='--')

# FG% conditional on streak going in
streak_fg = df.groupby('streak')['shot_made'].mean()
streak_fg_count = df.groupby('streak')['shot_made'].count()
# Filter to streaks with enough sample (>= 15 shots)
streak_fg = streak_fg[streak_fg_count >= 15]

axes[1].bar(streak_fg.index, streak_fg.values * 100,
            color=['#d7191c' if x < 0 else '#2b83ba' if x > 0 else '#aaaaaa'
                   for x in streak_fg.index],
            edgecolor='white')
axes[1].axhline(overall_fg * 100, color='black', linewidth=2,
                linestyle='--', label=f'Season avg ({overall_fg*100:.1f}%)')
axes[1].set_xlabel('Streak entering shot')
axes[1].set_ylabel('FG% on this shot (%)')
axes[1].set_title('FG% Conditional on Prior Streak\n(Hot Hand test — rational view)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.suptitle("Luka Doncic 2025-26 — Streak Analysis", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nNote: if FG% after a hot streak is NOT significantly higher than baseline,')
print('that supports the Hot Hand being a fallacy (Gilovich et al., 1985)')

## 3. Run the Rational Engine — Type 2 System (Task 11)

The Bayesian observer anchors to Luka's season FG% as the prior. After each shot, it updates using the Beta-Binomial conjugate model, resetting to the prior at each new game.

In [ ]:
# ── Initialize Rational Engine ────────────────────────────────────────────
# Prior is set to season totals — the most informed rational baseline

rational = RationalEngine(
    prior_makes  = total_fgm,
    prior_misses = total_fga - total_fgm
)

rational_beliefs_before = []  # belief BEFORE seeing each shot
rational_beliefs_after  = []  # belief AFTER seeing each shot

current_game = None

for _, row in df.iterrows():
    # Reset at new game boundary
    if row['game_id'] != current_game:
        rational.new_game()
        current_game = row['game_id']

    rational_beliefs_before.append(rational.current_belief)
    rational.update(int(row['shot_made']), metadata={'shot_number': row['shot_number']})
    rational_beliefs_after.append(rational.current_belief)

df['rational_belief_before'] = rational_beliefs_before
df['rational_belief_after']  = rational_beliefs_after

print('Rational Engine complete.')
print(f"Belief range: [{df['rational_belief_before'].min():.4f}, {df['rational_belief_before'].max():.4f}]")
print(f"Mean belief : {df['rational_belief_before'].mean():.4f}  (should be ≈ season FG% = {overall_fg:.4f})")

## 4. Run the Cognitive Engine — Type 1 System (Task 12)

The biased observer uses exponential decay weighting. We run it at three heuristic strength levels to show how different degrees of bias produce different divergences from the rational baseline.

In [ ]:
# ── Run at three heuristic strength levels ────────────────────────────────
# These will become the slider presets in the Streamlit dashboard

STRENGTH_LEVELS = {
    'low_bias'    : 0.3,   # Mildly biased observer
    'medium_bias' : 0.6,   # Moderately biased fan
    'high_bias'   : 0.9,   # Strongly biased (classic Hot Hand believer)
}

for label, strength in STRENGTH_LEVELS.items():
    engine = CognitiveEngine(
        prior_fg_pct       = overall_fg,
        heuristic_strength = strength,
        prior_anchor       = 0.3
    )

    beliefs_before = []
    current_game = None

    for _, row in df.iterrows():
        if row['game_id'] != current_game:
            engine.new_game()
            current_game = row['game_id']

        beliefs_before.append(engine.current_belief)
        engine.update(int(row['shot_made']))

    col_name = f'cognitive_belief_{label}'
    df[col_name] = beliefs_before

    print(f'{label} (strength={strength}): '
          f'mean={df[col_name].mean():.4f}, '
          f'std={df[col_name].std():.4f}, '
          f'range=[{df[col_name].min():.3f}, {df[col_name].max():.3f}]')

In [ ]:
# ── Compute Divergence Metric (Task 13 preview) ───────────────────────────
# Divergence = |cognitive_belief - rational_belief| at each shot
# This becomes the core output of the Bias Gap Graph

for label in STRENGTH_LEVELS:
    col = f'cognitive_belief_{label}'
    df[f'divergence_{label}'] = (df[col] - df['rational_belief_before']).abs()

print('Divergence metrics added.')
print('\nMean divergence by heuristic strength:')
for label, strength in STRENGTH_LEVELS.items():
    mean_div = df[f'divergence_{label}'].mean()
    print(f'  {label} (strength={strength}): {mean_div:.4f}  ({mean_div*100:.2f} percentage points avg drift)')

## 5. Belief Comparison Visualization

In [ ]:
# ── Pick one game to show a detailed belief trace ─────────────────────────
# Choose a game with a notable streak for visual impact
# Find the game with the longest consecutive make streak

game_max_streak = df.groupby('game_id')['streak'].max().idxmax()
game_df = df[df['game_id'] == game_max_streak].copy()
game_date_str = game_df['game_date'].iloc[0].strftime('%b %d, %Y')

print(f'Showing belief trace for game: {game_date_str} (highest make streak in season)')
print(f'Shots in game: {len(game_df)}')
print(f'Max streak in game: +{game_df["streak"].max()}')

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

shot_nums = game_df['game_shot_number']

# ── Top: Belief traces ────────────────────────────────────────────────────
axes[0].plot(shot_nums, game_df['rational_belief_before'] * 100,
             color='#2166ac', linewidth=2.5, label='Rational (Bayesian)', zorder=3)
axes[0].plot(shot_nums, game_df['cognitive_belief_high_bias'] * 100,
             color='#d7191c', linewidth=2.5, linestyle='--',
             label='Cognitive (High Bias, strength=0.9)', zorder=3)
axes[0].plot(shot_nums, game_df['cognitive_belief_medium_bias'] * 100,
             color='#f4a442', linewidth=1.5, linestyle=':',
             label='Cognitive (Medium Bias, strength=0.6)', zorder=2)
axes[0].axhline(overall_fg * 100, color='gray', linewidth=1,
                linestyle='-.', label=f'Season FG% ({overall_fg*100:.1f}%)')

# Mark made/missed shots along x-axis
for _, row in game_df.iterrows():
    color = '#27ae60' if row['shot_made'] == 1 else '#c0392b'
    axes[0].axvline(row['game_shot_number'], color=color, alpha=0.15, linewidth=1)

axes[0].set_ylabel('Estimated FG Probability (%)')
axes[0].set_title(f'Rational vs. Biased Belief — {game_date_str}\n'
                  '(green lines = makes, red lines = misses)')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].legend(loc='upper right', fontsize=10)
axes[0].set_ylim(20, 80)

# ── Bottom: Divergence (Bias Gap) ────────────────────────────────────────
axes[1].fill_between(shot_nums, game_df['divergence_high_bias'] * 100,
                     alpha=0.6, color='#d7191c', label='High Bias divergence')
axes[1].fill_between(shot_nums, game_df['divergence_medium_bias'] * 100,
                     alpha=0.4, color='#f4a442', label='Medium Bias divergence')
axes[1].set_xlabel('Shot number within game')
axes[1].set_ylabel('|Bias Gap| (percentage points)')
axes[1].set_title('Divergence from Rational Baseline (Bias Gap)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── Season-level divergence by streak magnitude ───────────────────────────
# Key analysis: does divergence grow as streak length increases?
# This is the "tipping point" your proposal expects to find around streak=3

streak_div = df.groupby('streak').agg(
    high_div   = ('divergence_high_bias',   'mean'),
    medium_div = ('divergence_medium_bias', 'mean'),
    low_div    = ('divergence_low_bias',    'mean'),
    count      = ('shot_made',              'count')
).query('count >= 15')  # only streaks with meaningful sample

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(streak_div.index, streak_div['high_div']   * 100, 'o-', color='#d7191c',
        linewidth=2.5, markersize=7, label='High Bias (strength=0.9)')
ax.plot(streak_div.index, streak_div['medium_div'] * 100, 's-', color='#f4a442',
        linewidth=2.5, markersize=7, label='Medium Bias (strength=0.6)')
ax.plot(streak_div.index, streak_div['low_div']    * 100, '^-', color='#74add1',
        linewidth=2.5, markersize=7, label='Low Bias (strength=0.3)')

ax.axvline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax.axvline(3, color='purple', linewidth=1.5, linestyle=':', alpha=0.7,
           label='Hypothesized tipping point (+3 streak)')

ax.set_xlabel('Streak entering shot (+ = consecutive makes, - = consecutive misses)')
ax.set_ylabel('Mean |Bias Gap| (percentage points)')
ax.set_title('Divergence from Rational Baseline by Streak Magnitude\n'
             'Does the cognitive bias amplify with longer streaks?')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend()

plt.tight_layout()
plt.show()

print('\nIf divergence increases monotonically as streak magnitude grows,')
print('that validates the Hot Hand cognitive model.')

## 6. Save Enriched CSV

In [ ]:
df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {df.shape}')
print(f'\nNew columns added:')
new_cols = [c for c in df.columns if any(x in c for x in
            ['rational', 'cognitive', 'divergence'])]
for col in new_cols:
    print(f'  {col}')